# Polaris RE — Reserve-Basis Comparison: Profit Signature Across Bases

**Purpose:** Demonstrate the `reserve_basis` selector (reserve-basis epic). A
reinsurer pricing an inforce block must reproduce the **cedant's** reserve, not
just the engine's historical net-premium reserve, because the reserve drives the
Net Amount at Risk (YRT), the proportional reserve transfer (coinsurance/modco),
and the **profit signature**.

This notebook prices the same whole-life block under each implemented basis and
compares the reserve run-off and the profit signature:

- **NET_PREMIUM** — the engine's historical net level premium reserve (default).
- **CRVM** — Commissioners Reserve Valuation Method (US statutory, full
  preliminary term), valued prospectively to omega.
- **VM20** — VM-20 simplified deterministic reserve, `max(NPR, DR)`.
- **GAAP** — US GAAP (FAS 60) net level premium benefit reserve on locked-in
  best-estimate assumptions plus explicit provisions for adverse deviation (PADs),
  valued prospectively to omega (section 5c).

It also illustrates the **whole-life terminal-reserve artefact** the epic
closes: the NET_PREMIUM whole-life reserve collapses at the projection horizon
edge (a one-period terminal estimate), while the CRVM / VM20 / GAAP bases grade
the reserve toward the face amount because they value to omega.

In [1]:
from datetime import date
from pathlib import Path

import numpy as np

from polaris_re.analytics.profit_test import ProfitTester
from polaris_re.assumptions.assumption_set import AssumptionSet
from polaris_re.assumptions.lapse import LapseAssumption
from polaris_re.assumptions.mortality import MortalityTable, MortalityTableSource
from polaris_re.core.inforce import InforceBlock
from polaris_re.core.policy import Policy, ProductType, Sex, SmokerStatus
from polaris_re.core.projection import ProjectionConfig
from polaris_re.core.reserve_basis import ReserveBasis
from polaris_re.products.dispatch import get_product_engine
from polaris_re.utils.table_io import MortalityTableArray

## 1. Build a whole-life inforce block

Whole life is the product where the reserve basis matters most: it carries a
material terminal reserve, and the net-premium path exhibits the horizon-edge
artefact. We use a small, homogeneous block (all male non-smoker, new issue) so
the comparison is easy to read.

In [2]:
VALUATION_DATE = date(2025, 1, 1)

policies = [
    Policy(
        policy_id=f"WL{i:03d}",
        issue_age=45,
        attained_age=45,
        sex=Sex.MALE,
        smoker_status=SmokerStatus.NON_SMOKER,
        underwriting_class="STANDARD",
        face_amount=250_000.0,
        annual_premium=6_000.0,
        policy_term=None,  # whole life pays to omega
        duration_inforce=0,
        issue_date=VALUATION_DATE,
        valuation_date=VALUATION_DATE,
        product_type=ProductType.WHOLE_LIFE,
    )
    for i in range(5)
]
block = InforceBlock(policies=policies)
print(f"{block.n_policies} policies, total face ${block.total_face_amount():,.0f}")

5 policies, total face $1,250,000


## 2. Assumptions (flat mortality + lapse)

A flat best-estimate table keeps the example self-contained (no CSV load). The
same projection mortality is used as the valuation mortality for CRVM/VM20 in
the current epic slices (the distinct 2001 CSO valuation table is a promoted
follow-up).

In [3]:
N_AGES = 121 - 18
qx = np.full(N_AGES, 0.008, dtype=np.float64).reshape(-1, 1)

tables = {}
for sex in (Sex.MALE, Sex.FEMALE):
    for smoker in (SmokerStatus.SMOKER, SmokerStatus.NON_SMOKER, SmokerStatus.UNKNOWN):
        tables[f"{sex.value}_{smoker.value}"] = MortalityTableArray(
            rates=qx.copy(), min_age=18, max_age=120, select_period=0,
            source_file=Path("synthetic"),
        )

mortality = MortalityTable(
    source=MortalityTableSource.CSO_2001, table_name="Synthetic flat",
    min_age=18, max_age=120, select_period_years=0,
    has_smoker_distinct_rates=False, tables=tables,
)
lapse = LapseAssumption.from_duration_table(
    {1: 0.05, 2: 0.05, 3: 0.05, "ultimate": 0.05}
)
assumptions = AssumptionSet(
    mortality=mortality, lapse=lapse, version="reserve-basis-nb",
    effective_date=VALUATION_DATE,
)

## 3. Project under each reserve basis

The only thing that changes between runs is `ProjectionConfig.reserve_basis`.
The treaty layer is omitted here so the comparison isolates the reserve effect on
the gross profit signature; the same basis would equally reprice the YRT NAR and
the coinsurance reserve transfer.

In [4]:
def price_under_basis(basis: ReserveBasis):
    config = ProjectionConfig(
        valuation_date=VALUATION_DATE,
        projection_horizon_years=20,
        discount_rate=0.06,
        maintenance_cost_per_policy_per_year=50.0,
        reserve_basis=basis,
    )
    engine = get_product_engine(inforce=block, assumptions=assumptions, config=config)
    gross = engine.project()
    profit = ProfitTester(cashflows=gross, hurdle_rate=0.10).run()
    return gross, profit


results = {basis: price_under_basis(basis) for basis in (
    ReserveBasis.NET_PREMIUM, ReserveBasis.CRVM, ReserveBasis.VM20
)}

## 4. Profit signature across bases

The reserve basis shifts the reserve run-off, which moves the timing of profit
(reserve increases are a cash strain) and the present value of profits.

In [5]:
header = f"{'Basis':<14}{'PV profits':>16}{'Undisc. profit':>18}{'Margin':>10}"
print(header)
print("-" * len(header))
for basis, (gross, profit) in results.items():
    margin = f"{profit.profit_margin:.2%}" if profit.profit_margin is not None else "N/A"
    print(
        f"{basis.value:<14}{profit.pv_profits:>16,.0f}"
        f"{profit.total_undiscounted_profit:>18,.0f}{margin:>10}"
    )

Basis               PV profits    Undisc. profit    Margin
----------------------------------------------------------
NET_PREMIUM            121,278           231,161    65.58%
CRVM                   118,162           224,294    63.90%
VM20                   118,162           224,294    63.90%


## 5b. CRVM on a *prescribed* statutory valuation table

Exact reproduction of a cedant's US statutory CRVM reserve values the reserve on the **prescribed** valuation mortality table (e.g. 2001 CSO), not the pricing best-estimate table. `AssumptionSet.valuation_mortality` (ADR-125) carries that prescribed table; when set, CRVM and the VM-20 NPR floor value on it. Here we compare CRVM valued on the best-estimate table (0.008) against a conservative prescribed table (flat 0.012). `NET_PREMIUM` is a historical pricing basis and ignores the slot.

On the CLI/API this is `--valuation-mortality CSO_2001` / `deal.valuation_mortality: "CSO_2001"` / the `valuation_mortality` request field — this cell uses a synthetic table so the notebook runs without the converted SOA/CSO CSVs.

In [ ]:
# A conservative *prescribed* valuation table (flat 0.012 vs the 0.008
# best-estimate). In production this is the 2001 CSO table loaded via
# MortalityTable.load(MortalityTableSource.CSO_2001, ...).
prescribed_qx = np.full(N_AGES, 0.012, dtype=np.float64).reshape(-1, 1)
prescribed_tables = {
    f"{sex.value}_{smoker.value}": MortalityTableArray(
        rates=prescribed_qx.copy(), min_age=18, max_age=120,
        select_period=0, source_file=Path("synthetic"),
    )
    for sex in (Sex.MALE, Sex.FEMALE)
    for smoker in (SmokerStatus.SMOKER, SmokerStatus.NON_SMOKER, SmokerStatus.UNKNOWN)
}
prescribed = MortalityTable(
    source=MortalityTableSource.CSO_2001,
    table_name="Prescribed valuation (flat 0.012)",
    min_age=18, max_age=120, select_period_years=0,
    has_smoker_distinct_rates=False, tables=prescribed_tables,
)
# Same best-estimate projection mortality; only the valuation table is added.
assumptions_stat = AssumptionSet(
    mortality=mortality, lapse=lapse, valuation_mortality=prescribed,
    version="reserve-basis-nb-stat", effective_date=VALUATION_DATE,
)

def price_basis(basis: ReserveBasis, assumption_set: AssumptionSet):
    config = ProjectionConfig(
        valuation_date=VALUATION_DATE, projection_horizon_years=20,
        discount_rate=0.06, maintenance_cost_per_policy_per_year=50.0,
        reserve_basis=basis,
    )
    engine = get_product_engine(inforce=block, assumptions=assumption_set, config=config)
    return ProfitTester(cashflows=engine.project(), hurdle_rate=0.10).run()

crvm_best = price_basis(ReserveBasis.CRVM, assumptions)
crvm_presc = price_basis(ReserveBasis.CRVM, assumptions_stat)
netp_best = price_basis(ReserveBasis.NET_PREMIUM, assumptions)
netp_presc = price_basis(ReserveBasis.NET_PREMIUM, assumptions_stat)

header = f"{'CRVM valued on':<26}{'PV profits':>16}"
print(header)
print('-' * len(header))
print(f"{'best-estimate (q=0.008)':<26}{crvm_best.pv_profits:>16,.0f}")
print(f"{'prescribed (q=0.012)':<26}{crvm_presc.pv_profits:>16,.0f}")

# A conservative prescribed table raises the CRVM reserve and so shifts the
# profit signature; NET_PREMIUM ignores valuation_mortality entirely.
assert abs(crvm_best.pv_profits - crvm_presc.pv_profits) > 1.0, \
    "prescribed valuation table must move the CRVM profit"
np.testing.assert_allclose(netp_best.pv_profits, netp_presc.pv_profits)
print("\nNET_PREMIUM ignores the prescribed table (identical PV): OK")

## 5c. GAAP (FAS 60) with provisions for adverse deviation (PADs)

US GAAP (FAS 60) values a traditional life reserve as a net **level** premium
benefit reserve on **locked-in best-estimate assumptions plus explicit provisions
for adverse deviation (PADs)**. Unlike the statutory bases (CRVM / VM-20 NPR), it
does **not** value on the prescribed `valuation_mortality` table — it is a
best-estimate + PAD basis, not a prescribed static one.

For whole life the GAAP reserve is valued **prospectively to omega** (like
CRVM/VM-20, so it does not collapse at the horizon edge), using a single net level
valuation premium (not CRVM's Full-Preliminary-Term split). Two knobs on
`ProjectionConfig` carry the margins:

- `gaap_mortality_pad` (default `1.0`) — a multiplicative margin on the
  best-estimate `q`.
- `gaap_interest_margin` (default `0.0`) — an absolute haircut to the valuation
  interest rate.

With both neutral, GAAP reduces to the locked-in best-estimate net level premium
reserve. A positive PAD raises the accumulation-phase reserve (more conservative).

In [ ]:
def price_gaap(mortality_pad: float, interest_margin: float):
    config = ProjectionConfig(
        valuation_date=VALUATION_DATE, projection_horizon_years=20,
        discount_rate=0.06, maintenance_cost_per_policy_per_year=50.0,
        reserve_basis=ReserveBasis.GAAP,
        gaap_mortality_pad=mortality_pad, gaap_interest_margin=interest_margin,
    )
    engine = get_product_engine(inforce=block, assumptions=assumptions, config=config)
    gross = engine.project()
    profit = ProfitTester(cashflows=gross, hurdle_rate=0.10).run()
    return gross, profit

# Neutral PADs (1.0 / 0.0) vs a conservative basis (mortality +10%, interest -100bp).
gaap_neutral_g, gaap_neutral_p = price_gaap(1.0, 0.0)
gaap_padded_g, gaap_padded_p = price_gaap(1.10, 0.01)

header = f"{'GAAP basis':<28}{'PV profits':>16}{'Reserve yr10':>16}"
print(header)
print('-' * len(header))
for label, (g, p) in (
    ("neutral PADs (1.0 / 0.0)", (gaap_neutral_g, gaap_neutral_p)),
    ("padded (x1.10 / -1.00%)", (gaap_padded_g, gaap_padded_p)),
):
    print(f"{label:<28}{p.pv_profits:>16,.0f}{g.reserve_balance[119]:>16,.0f}")

# A positive PAD raises the accumulation-phase reserve (a cash strain), lowering
# the PV of profits vs the neutral (locked-in best-estimate) basis.
assert gaap_padded_g.reserve_balance[119] > gaap_neutral_g.reserve_balance[119], \
    "a positive PAD must raise the accumulation-phase GAAP reserve"
print("\nGAAP is best-estimate + PAD; neutral PADs = the locked-in best estimate: OK")

## 5. Whole-life terminal-reserve artefact

The aggregate `reserve_balance` at years 10 and 20 shows the artefact directly:
under NET_PREMIUM the whole-life reserve collapses near the horizon edge (a
one-period terminal estimate), while CRVM and VM20 — valued prospectively to
omega — grade the reserve upward toward the face amount, as a permanent block's
reserve should.

In [6]:
print(f"{'Basis':<14}{'Reserve yr10':>16}{'Reserve yr20':>16}{'yr20/yr10':>12}")
print("-" * 58)
for basis, (gross, profit) in results.items():
    rb = gross.reserve_balance
    y10, y20 = rb[119], rb[-1]  # month 120 = end of year 10; final month = year 20
    ratio = f"{y20 / y10:.2f}x" if y10 > 0 else "n/a"
    print(f"{basis.value:<14}{y10:>16,.0f}{y20:>16,.0f}{ratio:>12}")

net_y20 = results[ReserveBasis.NET_PREMIUM][0].reserve_balance[-1]
crvm_y20 = results[ReserveBasis.CRVM][0].reserve_balance[-1]
print(
    f"\nCRVM yr20 reserve is {crvm_y20 / net_y20:.0f}x the NET_PREMIUM yr20 reserve "
    "— the artefact the reserve-basis epic closes."
)

Basis             Reserve yr10    Reserve yr20   yr20/yr10
----------------------------------------------------------
NET_PREMIUM                238             255       1.07x
CRVM                     4,148           7,122       1.72x
VM20                     4,148           7,122       1.72x

CRVM yr20 reserve is 28x the NET_PREMIUM yr20 reserve — the artefact the reserve-basis epic closes.


## 6. Takeaway

A reinsurer can now price the **same** block under the cedant's reserve basis by
flipping one selector — on the CLI (`polaris price --reserve-basis CRVM`), the
API (`reserve_basis` on the price request), or directly on `ProjectionConfig`.
The default remains NET_PREMIUM, so every existing run is unchanged.
